# 02 Fine-Tune PlantDoc Dataset

This notebook fine-tunes the previously trained PlantVillage MobileNetV2 model on the PlantDoc dataset.

Workflow:
1. Rename PlantDoc folders once to match PlantVillage class names.
2. Load PlantDoc train/test data.
3. Load `best_plant_disease_model.pth`.
4. Freeze most MobileNetV2 layers.
5. Fine-tune on PlantDoc using GPU.
6. Save `best_finetuned_plantdoc_model.pth`.

In [ ]:
import os
import copy
import time
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1. GPU Check

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    print("CUDA GPU not detected. Training will run on CPU and will be slow.")

## 2. Set Paths

Change these paths according to your computer.

Important:
- `PLANTDOC_ROOT` should contain `train` and `test`.
- `CHECKPOINT_PATH` should point to your PlantVillage trained model.

In [ ]:
PLANTDOC_ROOT = os.path.join(".", "Datasets", "PlantDoc")
CHECKPOINT_PATH = os.path.join(".", "best_plant_disease_model.pth")

train_dir = os.path.join(PLANTDOC_ROOT, "train")
test_dir = os.path.join(PLANTDOC_ROOT, "test")

print("Current Working Directory:", os.getcwd())
print("PlantDoc Root:", os.path.abspath(PLANTDOC_ROOT))
print("Train Directory:", os.path.abspath(train_dir))
print("Test Directory:", os.path.abspath(test_dir))

print("\nExists Check")
print("PlantDoc root:", os.path.exists(PLANTDOC_ROOT))
print("Train:", os.path.exists(train_dir))
print("Test:", os.path.exists(test_dir))
print("Checkpoint:", os.path.exists(CHECKPOINT_PATH))

## 3. Rename PlantDoc Folders Once

In [ ]:
''' class_mapping = {
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Apple leaf": "Apple___healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust",

    "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",

    "Blueberry leaf": "Blueberry___healthy",
    "Cherry leaf": "Cherry_(including_sour)___healthy",

    "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn rust leaf": "Corn_(maize)___Common_rust_",

    "Peach leaf": "Peach___healthy",

    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",

    "Raspberry leaf": "Raspberry___healthy",
    "Soyabean leaf": "Soybean___healthy",

    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Strawberry leaf": "Strawberry___healthy",

    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato leaf": "Tomato___healthy",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",

    "grape leaf": "Grape___healthy",
    "grape leaf black rot": "Grape___Black_rot",
}

def rename_plantdoc_folders(root):
    for split in ["train", "test"]:
        split_path = os.path.join(root, split)

        if not os.path.exists(split_path):
            print(f"Skipping missing folder: {split_path}")
            continue

        print(f"\nProcessing {split_path}")

        for old_name, new_name in class_mapping.items():
            old_path = os.path.join(split_path, old_name)
            new_path = os.path.join(split_path, new_name)

            if os.path.exists(old_path):
                if os.path.exists(new_path):
                    print(f"Already exists, not renaming: {new_name}")
                else:
                    os.rename(old_path, new_path)
                    print(f"Renamed: {old_name} --> {new_name}")

    print("\nFolder renaming finished.")

rename_plantdoc_folders(PLANTDOC_ROOT) '''

## 4. Load Checkpoint and PlantVillage Class Names

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

plantvillage_classes = checkpoint["class_names"]
num_classes = checkpoint["num_classes"]

print("Number of PlantVillage classes:", num_classes)
print("First 10 classes:", plantvillage_classes[:10])

## 5. Image Transforms

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.RandomAffine(
        degrees=15,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

## 6. Load PlantDoc Dataset

In [ ]:
train_dataset_full = datasets.ImageFolder(train_dir, transform=train_transforms)

print("PlantDoc train classes:", len(train_dataset_full.classes))
print(train_dataset_full.classes)

missing_from_checkpoint = [c for c in train_dataset_full.classes if c not in plantvillage_classes]

if missing_from_checkpoint:
    print("\nThese classes are not found in the PlantVillage checkpoint:")
    for c in missing_from_checkpoint:
        print("-", c)
    raise ValueError("Class mismatch found. Fix folder names before training.")

print("\nAll PlantDoc classes exist in the PlantVillage checkpoint.")

## 7. Force PlantDoc Labels to Use PlantVillage Indices

This is important because PlantDoc has fewer classes than PlantVillage.  
The model still has 38 output neurons, so labels must use the exact PlantVillage class indices.

In [ ]:
plantvillage_class_to_idx = {cls: idx for idx, cls in enumerate(plantvillage_classes)}

def remap_imagefolder_to_checkpoint_indices(dataset, checkpoint_class_to_idx):
    new_samples = []

    for path, old_idx in dataset.samples:
        old_class_name = dataset.classes[old_idx]
        new_idx = checkpoint_class_to_idx[old_class_name]
        new_samples.append((path, new_idx))

    dataset.samples = new_samples
    dataset.imgs = new_samples
    dataset.targets = [label for _, label in new_samples]
    dataset.classes = plantvillage_classes
    dataset.class_to_idx = checkpoint_class_to_idx

    return dataset

train_dataset_full = remap_imagefolder_to_checkpoint_indices(train_dataset_full, plantvillage_class_to_idx)

print("Remapping complete.")
print("Total training images:", len(train_dataset_full))

## 8. Create Validation Split from PlantDoc Train

In [ ]:
val_ratio = 0.2
val_size = int(len(train_dataset_full) * val_ratio)
train_size = len(train_dataset_full) - val_size

generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(train_dataset_full, [train_size, val_size], generator=generator)

# Important: validation should use validation transforms, not train augmentation.
valid_dataset.dataset.transform = valid_transforms

print("Train size:", len(train_dataset))
print("Validation size:", len(valid_dataset))

## 9. Load PlantDoc Test Dataset

In [ ]:
test_dataset = None

if os.path.exists(test_dir):
    test_dataset = datasets.ImageFolder(test_dir, transform=valid_transforms)

    missing_test = [c for c in test_dataset.classes if c not in plantvillage_classes]
    if missing_test:
        print("Test class mismatch:")
        for c in missing_test:
            print("-", c)
        raise ValueError("Test class mismatch found.")

    test_dataset = remap_imagefolder_to_checkpoint_indices(test_dataset, plantvillage_class_to_idx)
    print("Test size:", len(test_dataset))
else:
    print("No test folder found.")

## 10. DataLoaders

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = None
if test_dataset is not None:
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

## 11. Load MobileNetV2 and Checkpoint Weights

In [ ]:
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

print("PlantVillage checkpoint loaded successfully.")

## 12. Freeze Layers for Fine-Tuning

In [ ]:
# Freeze all feature extractor layers first
for param in model.features.parameters():
    param.requires_grad = True

# Unfreeze last few MobileNetV2 blocks for domain adaptation
for layer in model.features[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Always train classifier
for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

## 13. Loss, Optimizer, Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-5,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=2,
    factor=0.3
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type=="cuda"))

## 14. Training and Evaluation Functions

In [ ]:
def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type=="cuda")
):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, all_labels, all_preds


def fine_tune_model(model, epochs=40, patience=8):
    best_model_weights = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    patience_counter = 0

    history = {
        "train_loss": [],
        "train_acc": [],
        "valid_loss": [],
        "valid_acc": []
    }

    for epoch in range(epochs):
        start_time = time.time()

        train_loss, train_acc, _, _ = run_one_epoch(model, train_loader, optimizer)
        valid_loss, valid_acc, valid_labels, valid_preds = run_one_epoch(model, valid_loader)

        scheduler.step(valid_acc)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["valid_loss"].append(valid_loss)
        history["valid_acc"].append(valid_acc)

        elapsed = time.time() - start_time

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}")
        print(f"Time: {elapsed:.2f} sec")

        if valid_acc > best_acc:
            best_acc = valid_acc
            best_model_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0

            precision, recall, f1, _ = precision_recall_fscore_support(
                valid_labels,
                valid_preds,
                average="weighted",
                zero_division=0
            )

            torch.save({
     "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "class_names": plantvillage_classes,
    "num_classes": num_classes,
    "source_model": CHECKPOINT_PATH,
    "fine_tuned_on": "PlantDoc",
    "best_validation_accuracy": float(best_acc),
    "precision_weighted": float(precision),
    "recall_weighted": float(recall),
    "f1_weighted": float(f1),
    "epoch": epoch + 1
}, "best_finetuned_plantdoc_model_More_Epochs_40.pth")

            print("Best fine-tuned model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("\nEarly stopping triggered.")
            break

    model.load_state_dict(best_model_weights)
    print(f"\nBest validation accuracy: {best_acc:.4f}")

    return model, history

## 15. Start Fine-Tuning

In [ ]:
model, history = fine_tune_model(
    model,
    epochs=40,
    patience=8
)

## 16. Plot Training Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["valid_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fine-Tuning Loss Curve")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["valid_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fine-Tuning Accuracy Curve")
plt.legend()
plt.grid(True)
plt.show()

## 17. Final Validation Report

In [ ]:
valid_loss, valid_acc, valid_labels, valid_preds = run_one_epoch(model, valid_loader)

used_class_indices = sorted(set(valid_labels) | set(valid_preds))
used_class_names = [plantvillage_classes[i] for i in used_class_indices]

print("Validation Accuracy:", valid_acc)
print("\nClassification Report:")
print(classification_report(
    valid_labels,
    valid_preds,
    labels=used_class_indices,
    target_names=used_class_names,
    zero_division=0
))

cm = confusion_matrix(valid_labels, valid_preds, labels=used_class_indices)

plt.figure(figsize=(14, 12))
plt.imshow(cm, interpolation="nearest")
plt.title("Validation Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(used_class_names))
plt.xticks(tick_marks, used_class_names, rotation=90)
plt.yticks(tick_marks, used_class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 18. Optional Test Evaluation

In [ ]:
if test_loader is not None:
    test_loss, test_acc, test_labels, test_preds = run_one_epoch(model, test_loader)

    used_test_indices = sorted(set(test_labels) | set(test_preds))
    used_test_names = [plantvillage_classes[i] for i in used_test_indices]

    print("Test Accuracy:", test_acc)
    print("\nTest Classification Report:")
    print(classification_report(
        test_labels,
        test_preds,
        labels=used_test_indices,
        target_names=used_test_names,
        zero_division=0
    ))
else:
    print("No test loader available.")

## 19. Prediction Function

In [ ]:
def predict_image(image_path, model_path="best_finetuned_plantdoc_model.pth"):
    checkpoint = torch.load(model_path, map_location=device)

    loaded_model = models.mobilenet_v2(weights=None)
    loaded_model.classifier[1] = nn.Linear(
        loaded_model.classifier[1].in_features,
        checkpoint["num_classes"]
    )

    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()

    class_names = checkpoint["class_names"]

    image = Image.open(image_path).convert("RGB")
    image = valid_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = loaded_model(image)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, prediction = torch.max(probabilities, dim=1)

    predicted_class = class_names[prediction.item()]
    confidence_score = confidence.item() * 100

    return predicted_class, confidence_score


# Example:
# image_path = r"D:\Datasets\PlantDoc-Dataset\test\Tomato___Late_blight\some_image.jpg"
# predicted_class, confidence = predict_image(image_path)
# print(predicted_class, confidence)

## Notes

- This notebook keeps the PlantVillage classifier with 38 output classes.
- PlantDoc has fewer classes, so only matching classes are fine-tuned.
- The final saved model is `best_finetuned_plantdoc_model.pth`.
- Use this model in Flask exactly like the previous model, but load the new checkpoint file.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# =====================
# CHANGE IMAGE PATH HERE
# =====================
#IMAGE_PATH = r"D:\AgroGuide\agroguidemodelapi-main\Datasets\New-plant-diseases-dataset\test\test\AppleScab3.JPG"
IMAGE_PATH = r"5402963.jpg"
MODEL_PATH = "best_finetuned_plantdoc_model_More_Epochs_40.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load checkpoint (PyTorch 2.6+ fix)
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

# Load model
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    checkpoint["num_classes"]
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

# Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# Read image
image = Image.open(IMAGE_PATH).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

# Predict
with torch.no_grad():
    outputs = model(input_tensor)
    probabilities = torch.softmax(outputs, dim=1)

    confidence, prediction = torch.max(probabilities, dim=1)

predicted_class = checkpoint["class_names"][prediction.item()]
confidence = confidence.item() * 100

# Show image
plt.figure(figsize=(6,6))
plt.imshow(image)
plt.axis("off")
plt.title(f"{predicted_class}\nConfidence: {confidence:.2f}%")
plt.show()

print(f"Prediction : {predicted_class}")
print(f"Confidence : {confidence:.2f}%")

In [1]:
import torch

checkpoint = torch.load(
    "best_finetuned_plantdoc_model_More_Epochs_40.pth",
    map_location="cpu"
)

class_names = checkpoint["class_names"]

print(f"Total classes: {len(class_names)}\n")

for i, cls in enumerate(class_names, 1):
    print(f"{i:2d}. {cls}")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\legio\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [5]:
python -m jupyter --version

SyntaxError: invalid syntax (2932049906.py, line 1)